# Optimización Avanzada de Entrenamiento usando Lightning

Tu equipo ha sido encargado de optimizar la huella de memoria de tus modelos de aprendizaje profundo. Dado que la memoria de la GPU es uno de los recursos más costosos en el presupuesto de infraestructura, tu gerente ha solicitado soluciones para reducir el uso máximo de memoria sin comprometer el rendimiento del modelo.

Este laboratorio te guiará en la implementación y evaluación de dos potentes técnicas de optimización de memoria: **Entrenamiento de Precisión Mixta (Mixed Precision Training)** y **Acumulación de Gradientes (Gradient Accumulation)**. Estos métodos pueden reducir significativamente el consumo de memoria de la GPU manteniendo la precisión del modelo, lo que potencialmente conduce a ahorros sustanciales de costos en tu infraestructura de ML.

Más allá de las optimizaciones en sí, este cuaderno profundizará tu comprensión del framework Lightning. Verás cómo utilizar componentes principales como el `Trainer` y los `Callbacks` para estructurar y controlar tu proceso de entrenamiento.

Al final de este laboratorio, habrás construido un marco de trabajo completo para:

* **Construir un `Callback` personalizado** para medir de manera confiable métricas de rendimiento clave como el pico de memoria de la GPU y la precisión de validación.
* **Encapsular la lógica central de entrenamiento** configurando el `Trainer` de Lightning con parámetros eficientes en memoria.
* **Realizar un benchmark sistemático** de las diferentes técnicas frente a una línea base para recopilar datos empíricos.
* **Analizar las compensaciones (trade-offs)** entre el uso de memoria y el rendimiento del modelo para tomar decisiones basadas en datos.

## Imports

In [ ]:
import sys
import time
import warnings

# Redirect stderr to a black hole to catch other potential messages
class BlackHole:
    def write(self, message):
        pass
    def flush(self):
        pass
sys.stderr = BlackHole()

# Ignore Python-level UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.optim as optim
from lightning.pytorch.callbacks import Callback
from torch.utils.data import DataLoader
from torchmetrics import Accuracy
from torchvision import datasets, transforms

import helper_utils

torch.set_float32_matmul_precision('medium')
warnings.filterwarnings("ignore", category=UserWarning)

## Definición de los datos y el modelo con Lightning

Antes de poder realizar el benchmark de las diferentes técnicas de optimización, es necesario definir el pipeline de datos y la arquitectura del modelo. Al igual que en el laboratorio anterior, utilizarás `LightningDataModule` y `LightningModule` para configurar estos componentes necesarios.

### `LightningDataModule` para CIFAR-10

* Define la clase `CIFAR10DataModule` que se encarga de descargar, preparar y cargar el conjunto de datos CIFAR-10.

In [ ]:
class CIFAR10DataModule(pl.LightningDataModule):
    """Un LightningDataModule para el conjunto de datos CIFAR10."""

    def __init__(self, data_dir='./data', batch_size=128, num_workers=0):
        """
        Inicializa el DataModule.

        Args:
            data_dir (str): Directorio para almacenar los datos.
            batch_size (int): Número de muestras por lote (batch).
            num_workers (int): Número de subprocesos para la carga de datos.
        """
        # Llama al constructor de la clase padre (LightningDataModule).
        super().__init__()
        # Almacena la ruta del directorio de datos.
        self.data_dir = data_dir
        # Almacena el tamaño del lote para los DataLoaders.
        self.batch_size = batch_size
        # Almacena el número de procesos worker para la carga de datos.
        self.num_workers = num_workers
        # Define una secuencia de transformaciones para aplicar a las imágenes.
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def prepare_data(self):
        """Descarga el dataset CIFAR10 si aún no está presente."""
        
        # Descarga la partición de entrenamiento de CIFAR10.
        datasets.CIFAR10(self.data_dir, train=True, download=True)
        # Descarga la partición de prueba (test) de CIFAR10.
        datasets.CIFAR10(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        """
        Asigna los conjuntos de datos de entrenamiento/validación para su uso en los dataloaders.

        Args:
            stage (str, opcional): La etapa del entrenamiento (ej., 'fit', 'test').
                                   El Trainer de Lightning requiere este argumento, pero no se
                                   utiliza en esta implementación ya que la lógica de configuración
                                   es la misma para todas las etapas. Por defecto es None.
        """
        
        # Crea la instancia del dataset de entrenamiento y aplica las transformaciones.
        self.cifar_train = datasets.CIFAR10(self.data_dir, train=True, transform=self.transform)
        # Crea la instancia del dataset de validación (usando el set de prueba) y aplica transformaciones.
        self.cifar_val = datasets.CIFAR10(self.data_dir, train=False, transform=self.transform)
    
    def train_dataloader(self):
        """Devuelve el DataLoader para el conjunto de entrenamiento."""
        # El DataLoader gestiona el agrupamiento (batching), barajado (shuffling) y carga paralela.
        return DataLoader(self.cifar_train, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=True)

    def val_dataloader(self):
        """Devuelve el DataLoader para el conjunto de validación."""
        # Barajar no es necesario para el conjunto de validación.
        return DataLoader(self.cifar_val, batch_size=self.batch_size, num_workers=self.num_workers)

### `LightningModule` para el modelo CNN

* Define la clase `CIFAR10LightningModule`.
    * Utilizarás la configuración del modelo *eficiente* identificada en el laboratorio de perfilado (`conv_channels=(32, 64, 128)` y `linear_features=512`), ya que proporciona una línea base de rendimiento mucho mejor.

In [ ]:
class CIFAR10LightningModule(pl.LightningModule):
    """Un LightningModule flexible para la clasificación de imágenes CIFAR10."""

    def __init__(self, 
                 learning_rate=1e-3, 
                 weight_decay=0.01,
                 conv_channels=(32, 64, 128),
                 linear_features=512,
                 num_classes=10):
        """
        Inicializa el LightningModule con parámetros de capa configurables.

        Args:
            learning_rate: La tasa de aprendizaje para el optimizador.
            weight_decay: El decaimiento de peso (penalización L2) para el optimizador.
            conv_channels: Una tupla que especifica los canales de salida para cada
                           bloque convolucional.
            linear_features: El número de características en la capa oculta totalmente
                             conectada (fully connected).
            num_classes: El número de clases de salida para la tarea de clasificación.
        """
        # Llamar al constructor de la clase padre.
        super().__init__()
        # Guardar los hiperparámetros pasados al constructor.
        self.save_hyperparameters()
        
        # Calcular el tamaño aplanado de los mapas de características después de la
        # capa final de pooling. Esto es necesario para definir el tamaño de entrada
        # de la primera capa totalmente conectada.
        flattened_size = self.hparams.conv_channels[-1] * 4 * 4
        
        # Definir la arquitectura del modelo usando un contenedor secuencial.
        self.model = nn.Sequential(
            nn.Conv2d(3, self.hparams.conv_channels[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.hparams.conv_channels[0], self.hparams.conv_channels[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.hparams.conv_channels[1], self.hparams.conv_channels[2], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(flattened_size, self.hparams.linear_features),
            nn.ReLU(),
            nn.Linear(self.hparams.linear_features, self.hparams.num_classes)
        )
        
        # Inicializar la función de pérdida.
        self.loss_fn = nn.CrossEntropyLoss()
        
        # Inicializar métricas para rastrear la precisión de entrenamiento y validación.
        self.val_accuracy = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)

    def forward(self, x):
        """
        Define el paso hacia adelante (forward pass) del modelo.

        Args:
            x: El tensor de entrada que contiene un lote de imágenes.

        Returns:
            El tensor de salida (logits) del modelo.
        """
        # Pasar la entrada a través del modelo secuencial.
        return self.model(x)

    def training_step(self, batch, batch_idx=None):
        """
        Realiza un único paso de entrenamiento.
    
        Args:
            batch (Any): El lote de datos del dataloader.
            batch_idx (int, opcional): El índice del lote actual. El Trainer de Lightning
                                       requiere este argumento, pero no se utiliza en esta
                                       implementación. Por defecto es None.
        """
        # Desempaquetar el lote en entradas (imágenes) y etiquetas.
        inputs, labels = batch
        # Realizar un paso hacia adelante para obtener las predicciones del modelo (logits).
        outputs = self(inputs)
        # Calcular la pérdida.
        loss = self.loss_fn(outputs, labels)

        # Registrar la pérdida de entrenamiento.
        self.log("train_loss", loss)
        
        # Devolver la pérdida a Lightning para la retropropagación (backpropagation).
        return loss

    def validation_step(self, batch, batch_idx=None):
        """
        Realiza un único paso de validación.
    
        Args:
            batch (Any): El lote de datos del dataloader.
            batch_idx (int, opcional): El índice del lote actual. El Trainer de Lightning
                                       requiere este argumento, pero no se utiliza en esta
                                       implementación. Por defecto es None.
        """
        # Desempaquetar el lote en entradas (imágenes) y etiquetas.
        inputs, labels = batch
        # Realizar un paso hacia adelante para obtener las predicciones del modelo (logits).
        outputs = self(inputs)
        # Calcular la pérdida.
        loss = self.loss_fn(outputs, labels)

        # Registrar la pérdida de validación.
        self.log("val_loss", loss, prog_bar=True)
        # Actualizar la métrica de precisión de validación con los resultados del lote actual.
        self.val_accuracy(outputs, labels)
        # Registrar la precisión de validación.
        self.log("val_accuracy", self.val_accuracy, prog_bar=True)

    def configure_optimizers(self):
        """
        Configura y devuelve el optimizador del modelo.

        Returns:
            Una instancia del optimizador.
        """
        # Crear y devolver el optimizador AdamW.
        return optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)

## Construcción del marco de trabajo para Benchmarking

Ahora que el modelo y los datos están definidos, es necesario construir un sistema para automatizar el proceso de ejecución y comparación de los diferentes experimentos de optimización. Esto implicará la creación de un conjunto de funciones y componentes reutilizables que trabajen en conjunto para gestionar todo, desde la medición del rendimiento hasta la ejecución del bucle de entrenamiento.

### Creación de un Callback personalizado para la medición

Para medir el rendimiento de cada experimento de manera confiable, necesitas una forma consistente de recolectar datos. Lightning ofrece una solución elegante para esto llamada <code>[Callbacks](https://lightning.ai/docs/pytorch/stable/extensions/callbacks.html)</code>. Un `Callback` es un objeto que puede conectarse al proceso de entrenamiento en varios puntos (como el inicio de una época o el final de una ejecución) para ejecutar tu código personalizado. Esta es una herramienta esencial para el registro de datos (logging), el monitoreo o, en este caso, la medición del rendimiento.

Crearás una clase `PerformanceCallback` para recopilar todas las métricas necesarias para el análisis:
* **Uso máximo de memoria (Peak memory usage)**: La memoria máxima de la GPU utilizada en cualquier punto individual durante el entrenamiento.
* **Precisión de validación (Validation accuracy)**: El rendimiento del modelo en el conjunto de datos de validación no visto.

* Define `PerformanceCallback`:
    * `__init__`: El constructor donde inicializarás las variables para almacenar los datos que recolectes.
    * `on_train_start` y `on_train_end`: Estos ganchos (hooks) se utilizan para el monitoreo de memoria. `on_train_start` reinicia el contador de memoria pico de la GPU, y `on_train_end` registra el uso final de la memoria pico.
    * `on_validation_epoch_end`: Este gancho se ejecuta después de cada época de validación para guardar la precisión de validación más reciente.

In [ ]:
class PerformanceCallback(Callback):
    """
    Un Callback de Lightning para recolectar métricas de rendimiento clave durante el ciclo de vida
    de entrenamiento y validación.

    Este callback mide:
    - Precisión de validación
    - Uso máximo (pico) de memoria de la GPU durante el entrenamiento
    """

    def __init__(self):
        """Inicializa el almacenamiento para las métricas de rendimiento que se recolectarán."""
        # Inicializa una lista para almacenar las métricas de rendimiento
        self.metrics = []
        
        # Inicializa un atributo para almacenar el uso de memoria pico
        self.peak_memory_mb = None

    def on_train_start(self, trainer, pl_module=None):
        """
        Reinicia las estadísticas de memoria pico de la GPU al inicio del entrenamiento.
    
        Args:
            trainer (pl.Trainer): La instancia principal del Trainer de Lightning.
            pl_module (pl.LightningModule, opcional): El LightningModule que se está entrenando.
                                                      Requerido por el gancho del callback pero no
                                                      se utiliza aquí. Por defecto es None.
        """
        # Verifica si CUDA está disponible para operaciones de GPU
        if torch.cuda.is_available():
            # Reinicia las estadísticas de memoria pico en el dispositivo raíz
            torch.cuda.reset_peak_memory_stats(trainer.strategy.root_device)
            # Limpia el caché de CUDA para liberar memoria
            torch.cuda.empty_cache()

    def on_validation_epoch_end(self, trainer, pl_module=None):
        """
        Agrega y registra métricas al final de una época de validación.
    
        Args:
            trainer (pl.Trainer): La instancia del Trainer de Lightning.
            pl_module (pl.LightningModule, opcional): El LightningModule que se está validando.
                                                      Requerido por el gancho del callback pero no
                                                      se utiliza aquí. Por defecto es None.
        """
        # Verifica si el trainer no está en modo de verificación de cordura (sanity checking)
        if not trainer.sanity_checking:
            # Obtiene las métricas recolectadas por los callbacks del trainer
            metrics = trainer.callback_metrics
            # Añade un diccionario con la precisión de validación a la lista de métricas
            self.metrics.append({
                "val_accuracy": metrics["val_accuracy"].item() * 100
            })

    def on_train_end(self, trainer, pl_module=None):
        """
        Registra el uso de memoria pico de la GPU al finalizar el entrenamiento.
    
        Args:
            trainer (pl.Trainer): La instancia del Trainer de PyTorch Lightning.
            pl_module (pl.LightningModule, opcional): El LightningModule que fue entrenado.
                                                      Requerido por el gancho del callback pero no
                                                      se utiliza aquí. Por defecto es None.
        """
        # Verifica si CUDA está disponible para operaciones de GPU
        if torch.cuda.is_available():
            # Obtiene la memoria máxima asignada en el dispositivo raíz
            peak_memory = torch.cuda.max_memory_allocated(trainer.strategy.root_device)
            # Convierte la memoria pico de bytes a megabytes
            self.peak_memory_mb = peak_memory / 1024**2

### La función central de entrenamiento

Para ejecutar tus experimentos de optimización de manera limpia y repetible, encapsularás la lógica principal de entrenamiento en una función dedicada. El trabajo principal de esta función es configurar el <code>[Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html)</code> de Lightning, el componente responsable de gestionar todo el proceso de entrenamiento.

Para mantener el código organizado y enfatizar esta lógica central, definirás la configuración del `Trainer` en una función independiente llamada `run_training`. Cada experimento de optimización que realices más adelante llamará a esta misma función, solo que con diferentes ajustes de configuración.

#### La ventaja de Lightning

El `Trainer` de Lightning es el protagonista; automatiza y gestiona completamente todo el proceso. Este **es** tu bucle de entrenamiento. Puedes olvidarte de escribir bucles `for`, mover datos manualmente a la GPU o llamar a `.zero_grad()`, `.backward()` y `.step()`. El Trainer lo orquestación todo. Simplemente lo configuras y llamas a `.fit()`.

#### Encapsulando la ejecución del entrenamiento

* Define la función `run_training`, que contiene la lógica completa para una ejecución de entrenamiento. Inicializa un `Trainer`, ejecuta `.fit()` y luego devuelve el objeto `trainer`, que contiene todo el estado y los resultados del entrenamiento.
    * `precision`: Controla la precisión numérica para el entrenamiento (por ejemplo, `'16-mixed'` para habilitar la precisión mixta).
    * `accumulate_grad_batches`: Especifica el número de lotes a procesar antes de actualizar los pesos del modelo, permitiendo la acumulación de gradientes.
    * `callbacks`: Recibe una lista de objetos `Callback`. Aquí pasarás tu `PerformanceCallback` para recolectar métricas durante la ejecución.
* Devolver el `trainer` es importante porque contiene todo el estado y los resultados de la ejecución.

In [ ]:
def run_training(model, data_module, num_epochs, precision, grad_accum, performance_callback):
    """
    Configura y ejecuta un proceso de entrenamiento de Lightning.

    Args:
        model (pl.LightningModule): El modelo a ser entrenado.
        data_module (pl.LightningDataModule): El módulo de datos que proporciona los datasets.
        num_epochs (int): El número total de épocas para el entrenamiento.
        precision (str): La precisión numérica para el entrenamiento ('32-true', '16-mixed').
        grad_accum (int): El número de lotes (batches) sobre los cuales acumular gradientes.
        performance_callback (pl.Callback): Un callback para medir métricas de rendimiento.

    Returns:
        pl.Trainer: La instancia del trainer después de que el ajuste (fitting) se ha completado.
    """
    
    # Inicializa un Trainer de Lightning con parámetros específicos
    trainer = pl.Trainer(
        max_epochs=num_epochs,               # Establece el número máximo de épocas de entrenamiento
        accelerator="auto",                  # Selecciona automáticamente el mejor acelerador (ej. CPU, GPU)
        devices=1,                           # Usa un solo dispositivo para el entrenamiento
        precision=precision,                 # Establece la precisión numérica para el proceso de entrenamiento
        accumulate_grad_batches=grad_accum,  # Configura el número de lotes para acumular gradientes antes de actualizar pesos
        callbacks=[performance_callback],    # Proporciona una lista de callbacks para usar durante el entrenamiento
        logger=False,                        # Desactiva el registro (logging) para esta ejecución de entrenamiento
        enable_progress_bar=True,            # Activa la barra de progreso para mostrar el avance del entrenamiento
        enable_model_summary=False,          # Desactiva el resumen del modelo
        enable_checkpointing=False           # Desactiva la creación de puntos de control (checkpoints)
    )
    
    # Inicia el proceso de entrenamiento utilizando el modelo y el módulo de datos proporcionados
    trainer.fit(model, data_module)
    
    # Devuelve la instancia del trainer configurada y entrenada
    return trainer

### Orquestación de los experimentos

Ahora crearás la función principal `run_optimization`. Esta función gestionará todas las partes necesarias para cada experimento de principio a fin. Se encargará de:

* **Configurar el experimento**: Crear nuevas instancias de tu `model` y del `PerformanceCallback`.
* **Ejecutar el entrenamiento**: Llamar a la función `run_training` para gestionar el proceso de entrenamiento real.
* **Procesar los resultados**: Extraer las métricas de tu callback y formatear todo en un diccionario.
* **Devolver los hallazgos**: Retornar los resultados formateados para su posterior comparación y graficación.

In [ ]:
def run_optimization(name, data_module, num_epochs, precision, grad_accum):
    """
    Orquestra y ejecuta un experimento de optimización único y completo.

    Args:
        name (str): El nombre a mostrar para el experimento (ej., "Mixed Precision").
        data_module (pl.LightningDataModule): El módulo de datos para entrenamiento y validación.
        num_epochs (int): El número de épocas durante las cuales entrenar.
        precision (str): La precisión de entrenamiento a utilizar ('32-true', '16-mixed').
        grad_accum (int): El número de lotes para la acumulación de gradientes.
        
    Returns:
        tuple[dict, dict]: Una tupla que contiene dos diccionarios:
        1. Los resultados resumidos para la tabla comparativa.
        2. La información detallada requerida para generar gráficos.
    """
    
    # Configuración del experimento
    print(f"\n--- Ejecutando Experimento: {name} (Durante {num_epochs} Épocas) ---")
    
    model = CIFAR10LightningModule()
    performance_callback = PerformanceCallback()

    # Ejecución del entrenamiento
    trainer = run_training(
        model=model,
        data_module=data_module,
        num_epochs=num_epochs,
        precision=precision,
        grad_accum=grad_accum,
        performance_callback=performance_callback,
    )

    # Procesamiento de los resultados
    # Extrae la lista de precisiones de validación recolectadas por el callback
    accuracies = [m["val_accuracy"] for m in performance_callback.metrics]
    
    current_results = {
        "optimization": name,
        "final_acc": accuracies[-1],
        "peak_mem_mb": performance_callback.peak_memory_mb,
    }
    
    # Prepara la información para graficar combinando resultados y el historial de precisión
    plot_info = {**current_results, "accuracies": accuracies}
    
    # Devolución de los hallazgos
    return current_results, plot_info

### Ejecución de los experimentos de optimización

Con todas las funciones de ejecución listas, es hora de poner en marcha los experimentos. Primero, definirás las configuraciones compartidas para todas las próximas pruebas experimentales:

* `num_epochs`: Establece el número de épocas para cada ejecución de optimización.
* `batch_size` y `num_workers`: Utilizarás un `batch_size=256` y `num_workers=4` efectivo para mantenerte dentro de los límites de memoria de este entorno.
* `results[]`: Una lista vacía para recopilar los resultados finales resumidos de cada experimento.
* `plot_data[]`: Una segunda lista vacía para almacenar los datos necesarios para las visualizaciones.
* `data_module`: Una instancia de tu `CIFAR10DataModule`, configurada con el `batch_size` y el `num_workers` que acabas de definir.

In [ ]:
# Establece el número de épocas para cada ejecución
num_epochs = 5
# Define el número de muestras que se procesarán en cada lote (batch)
batch_size = 256
# Establece el número de procesos paralelos para la carga de datos
num_workers = 4

# Inicializa una lista vacía para almacenar los resultados resumidos finales para la tabla comparativa
results = []
# Inicializa una lista vacía para almacenar datos detallados para la generación de gráficos
plot_data = []

# Crea una instancia del DataModule, configurándolo con el tamaño de lote y el número de workers especificados
data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

Ahora verás las diferentes optimizaciones de entrenamiento en acción. Vas a realizar varios experimentos para comparar directamente sus efectos. Es importante recordar que ninguna técnica es universalmente mejor que otra. Cada optimización presenta un equilibrio distinto de beneficios e inconvenientes, lo que la hace adecuada para diferentes escenarios. Tu elección dependerá de lo que priorices para tu proyecto, como alcanzar la máxima precisión o el uso más eficiente de los recursos.

La siguiente tabla resume los diferentes experimentos. Ten en cuenta que todos los experimentos tienen el mismo `Effective Batch Size` (Tamaño de Lote Efectivo) de 256.

| Nombre | Effective Batch Size | Batch Size | Gradient Accumulation | Precisión |
| :--- | :--- | :--- | :--- | :--- |
| Standard | 256 | 256 | 1 | 32-bit |
| Mixed Precision | 256 | 256 | 1 | 16-bit |
| Gradient Accumulation (BS Efectivo: 256-128) | 256 | 128 | 2 | 32-bit |
| Gradient Accumulation (BS Efectivo: 256-64) | 256 | 64 | 4 | 32-bit |
| Combined (BS Efectivo: 256-128) | 256 | 128 | 2 | 16-bit |
| Combined (BS Efectivo: 256-64) | 256 | 64 | 4 | 16-bit |

#### Entrenamiento Estándar (Standard Training)

Esta primera ejecución establece tu línea base (baseline). Es el enfoque estándar y más directo para entrenar un modelo. Ejecutarás un proceso de entrenamiento sin ninguna optimización avanzada, lo que proporciona un punto de referencia esencial para medir de manera justa la efectividad de las otras técnicas.

Deberías considerar este como el método por defecto para cualquier proyecto nuevo. Es la mejor opción cuando tu objetivo principal es garantizar la máxima estabilidad numérica y precisión, y cuando no tienes limitaciones de tiempo de entrenamiento o memoria de GPU disponible.

El beneficio principal es su alta precisión, asegurando que todos los cálculos sean muy exactos y tus resultados sean confiables. Sin embargo, su principal inconveniente es que consume muchos recursos. Este método utiliza la mayor cantidad de memoria y puede ser significativamente más lento que otras técnicas, especialmente en hardware moderno diseñado para acelerar cálculos de menor precisión.



**Precisión Estándar (`32-true`)**

Este ajuste le indica al `Trainer` que realice todos los cálculos utilizando **números de punto flotante de 32 bits completos**. Este es el tipo de dato por defecto para la mayoría de las operaciones de aprendizaje profundo y ofrece un alto grado de exactitud numérica.

* **precision**: Configurado en `"32-true"`.
* **grad_accum**: Configurado en `1`, lo que significa que los pesos del modelo se actualizarán después de cada lote individual. Este es el comportamiento normal de entrenamiento sin acumulación.

In [ ]:
res, p_data = run_optimization(
    name="Standard",
    precision="32-true",
    grad_accum=1,
    data_module=data_module,
    num_epochs=num_epochs,
)

results.append(res)
plot_data.append(p_data)

#### Entrenamiento de Precisión Mixta (Mixed Precision Training)

En esta segunda ejecución, explorarás el **Entrenamiento de Precisión Mixta**. Esta técnica combina inteligentemente el uso de dos precisiones numéricas diferentes. Su objetivo es acelerar el entrenamiento y reducir el uso de memoria de la GPU realizando muchos cálculos utilizando un formato rápido de **punto flotante de 16 bits**. Para mantener la estabilidad numérica, mantiene estratégicamente las operaciones críticas, como las actualizaciones de pesos, en el formato estándar de 32 bits.

Deberías usar esta técnica cuando tu objetivo sea **entrenar más rápido** o **reducir la huella de memoria** de tu modelo. Los beneficios de velocidad son especialmente notables en las GPUs modernas con hardware diseñado para acelerar las matemáticas de menor precisión. Es una excelente opción para entrenar modelos muy grandes que, de otro modo, podrían no caber en la memoria de tu GPU. El posible inconveniente de esta técnica es un ligero riesgo de reducción de la precisión final debido a la menor precisión utilizada en algunos cálculos.


**Precisión Mixta (`16-mixed`)**

Al configurar la precisión en `"16-mixed"`, le indicas a Lightning que gestione el proceso automáticamente. Utilizará estratégicamente media precisión (16 bits) para operaciones como las multiplicaciones de matrices y precisión completa (32 bits) para operaciones donde se necesita una alta exactitud, como las actualizaciones de pesos.

* **precision**: Configurado en `"16-mixed"` para habilitar el entrenamiento automático de precisión mixta.
* **grad_accum**: Se mantiene en `1`, ya que en esta ejecución solo estás midiendo el efecto de la precisión mixta.

In [ ]:
res, p_data = run_optimization(
    name="Mixed Precision",
    precision="16-mixed",
    grad_accum=1,
    data_module=data_module,
    num_epochs=num_epochs,
)

results.append(res)
plot_data.append(p_data)

#### Acumulación de Gradientes (Gradient Accumulation)

A veces deseas obtener los beneficios de reducir el tamaño del lote (batch size) para disminuir el uso de memoria de la GPU sin comprometer la precisión. La **Acumulación de Gradientes** es la solución exacta para este problema. Esta técnica funciona procesando varios lotes más pequeños uno tras otro y sumando sus gradientes. Solo después de un número determinado de estos lotes pequeños, el modelo realiza una única actualización de pesos, lo que simula eficazmente el entrenamiento con un tamaño de lote mucho mayor sin el alto costo de memoria.



Deberías usar esta técnica cuando el entrenamiento sea inestable y creas que un tamaño de lote más grande ayudaría, pero estés limitado por la memoria de tu hardware. Es una estrategia muy común y efectiva para entrenar modelos grandes, como los Transformers, que a menudo requieren lotes grandes para converger correctamente.

El beneficio principal es lograr actualizaciones de entrenamiento más estables, lo que puede conducir a una mejor convergencia del modelo, todo ello utilizando menos memoria de la que requeriría un lote grande real. El principal inconveniente, sin embargo, es que puede aumentar el tiempo total de entrenamiento. Además, se debe tener precaución al usar esta técnica con capas como **Batch Normalization**, ya que a veces puede afectar su rendimiento.

**Tamaño de Lote Efectivo (Effective Batch Size)**

Esta métrica indica el tamaño del lote que estás simulando. La fórmula es:

> Tamaño de Lote Efectivo = batch_size * grad_accum

Con un `batch_size` de 128 y un `grad_accum` de 2, tu tamaño de lote efectivo es 256.

* **precision**: Configurado en `"32-true"` para medir únicamente el efecto de la acumulación de gradientes.
* **grad_accum**: Configurado en `2`. Esto le indica al `Trainer` que procese 2 lotes antes de actualizar los pesos del modelo.

In [ ]:
# Configuración del módulo de datos
batch_size = 128
data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

# Configuración de la acumulación de gradientes
effective_batch_size = 256
grad_accum = effective_batch_size // batch_size

# Ejecución del experimento
res, p_data = run_optimization(
    name="Gradient Accumulation (Effective BS: 256-128)",
    precision="32-true",
    grad_accum=grad_accum,
    data_module=data_module,
    num_epochs=num_epochs,
)

# Almacenamiento de resultados para comparación y graficación
results.append(res)
plot_data.append(p_data)

<br>

**Tu tarea**

Ahora, realiza otra ejecución de acumulación de gradientes, con el mismo tamaño de lote efectivo de 256 pero con un tamaño de lote diferente.

* **precision**: Para asegurarte de que solo estás midiendo el impacto de la acumulación de gradientes, ¿qué ajuste de precisión debería utilizarse? (*Pista*: no es precisión mixta)

* **grad_accum**: Dado un `batch_size` de 64, ¿qué valor de `grad_accum` deberías usar para alcanzar el objetivo de 256?

In [ ]:
try:
    # Configuración del módulo de datos
    batch_size = 64
    data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

    res, p_data = run_optimization(
        name="Gradient Accumulation (Effective BS: 256-64)",

        # Establece la precisión para la acumulación de gradientes
        precision=None, ### Añade tu código aquí
        # Establece los pasos de acumulación para lograr un tamaño de lote efectivo de 256
        grad_accum=None, ### Añade tu código aquí

        data_module=data_module,
        num_epochs=num_epochs,
    )

    results.append(res)
    plot_data.append(p_data)

except Exception as e:
    print("\033[91m¡Algo salió mal, inténtalo de nuevo!")
    raise e

<br>

<details>
<summary><span style="color:green;"><strong>Solution (Click here to expand)</strong></span></summary>

```python
try:
    # Data module setup
    batch_size = 64
    data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

    res, p_data = run_optimization(
        name="Gradient Accumulation (Effective BS: 256-64)",

        # Set precision for gradient accumulation
        precision="32-true",
        # Set the accumulation steps to achieve an effective batch size of 256
        grad_accum=4,

        data_module=data_module,
        num_epochs=num_epochs,
    )

    results.append(res)
    plot_data.append(p_data)

except Exception as e:
    print("\033[91mSomething went wrong, try again!")
    raise e
```

#### Combinación de Optimizaciones

Para la siguiente ejecución, combinarás la precisión mixta y la acumulación de gradientes. Este es un enfoque potente en el que utilizas ambas técnicas al mismo tiempo para crear una configuración de entrenamiento altamente eficiente. El objetivo es ver si puedes obtener los beneficios sinérgicos de ambas trabajando en conjunto.

Deberías considerar esta estrategia combinada cuando te enfrentes a restricciones significativas en la memoria de la GPU. Es especialmente útil para entrenar modelos de vanguardia (state-of-the-art) muy grandes que requieren un tamaño de lote elevado para una convergencia estable, pero que son demasiado grandes para caber en la memoria utilizando la precisión estándar.

El beneficio principal es lograr la máxima eficiencia de recursos. La precisión mixta reduce la huella de memoria y acelera el cálculo, mientras que la acumulación de gradientes te permite entrenar con un tamaño de lote efectivo grande y estable. El posible inconveniente es que los riesgos de cada técnica se combinan. Debes tener en cuenta una posible caída en la precisión debido a la precisión mixta, junto con la posibilidad de tiempos de entrenamiento más largos por la acumulación de gradientes. Como siempre, realizar un benchmark es esencial para confirmar que esta combinación produce el mejor resultado general para tu modelo y objetivo específicos.

* **precision**: Configurado en `"16-mixed"` para aprovechar la velocidad del cálculo en media precisión.
* **grad_accum**: Configurado en `2` para simular un tamaño de lote de 256.

In [ ]:
# Configuración del módulo de datos
batch_size = 128
data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

# Configuración de la acumulación de gradientes
effective_batch_size = 256
grad_accum = effective_batch_size // batch_size

# Ejecución de la optimización combinada (Precisión Mixta + Acumulación de Gradientes)
res, p_data = run_optimization(
    name="Combined (Effective BS: 256-128)",
    precision="16-mixed",
    grad_accum=grad_accum,
    data_module=data_module,
    num_epochs=num_epochs,
)

# Almacenamiento de los resultados y datos para graficar
results.append(res)
plot_data.append(p_data)

<br>

**Tu tarea**

Para el experimento final, combinarás ambas optimizaciones nuevamente, pero esta vez con un tamaño de lote de 64 para obtener un tamaño de lote efectivo de 256.

* **precision**: Dado que se trata de una ejecución de optimización combinada, ¿qué ajuste de precisión deberías usar para obtener los beneficios de memoria?

* **grad_accum**: Con un `batch_size` de 64, ¿qué valor de `grad_accum` deberías emplear para alcanzar el objetivo de un tamaño de lote efectivo de 256?

In [ ]:
try:
    # Configuración del módulo de datos
    batch_size = 64
    data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

    res, p_data = run_optimization(
        name="Combined (Effective BS: 256-64)",

        # Usa el ajuste para precisión mixta
        precision=None, ### Añade tu código aquí
        # Establece los pasos de acumulación para lograr un tamaño de lote efectivo de 256
        grad_accum=None, ### Añade tu código aquí    
        
        data_module=data_module,
        num_epochs=num_epochs,
    )

    results.append(res)
    plot_data.append(p_data)

except Exception as e:
    print("\033[91m¡Algo salió mal, inténtalo de nuevo!")
    raise e

<br>

<details>
<summary><span style="color:green;"><strong>Solution (Click here to expand)</strong></span></summary>

```python
try:
    # Data module setup
    batch_size = 64
    data_module = CIFAR10DataModule(batch_size=batch_size, num_workers=num_workers)

    res, p_data = run_optimization(
        name="Combined (Effective BS: 256-64)",

        # Use the setting for mixed precision
        precision="16-mixed",
        # Set the accumulation steps to achieve an effective batch size of 256
        grad_accum=4,
        
        data_module=data_module,
        num_epochs=num_epochs,
    )

    results.append(res)
    plot_data.append(p_data)

except Exception as e:
    print("\033[91mSomething went wrong, try again!")
    raise e
```

### Análisis de los Resultados

Después de ejecutar todos los experimentos, el paso final es mostrar y visualizar los datos recolectados para comprender los compromisos (trade-offs) entre las diferentes estrategias.

* En primer lugar, una tabla de resumen proporcionará una visión numérica detallada de las métricas clave de cada ejecución.

In [ ]:
# Display the comparison table of all experiment results
helper_utils.optimization_results(results)

**Abreviaturas de las técnicas de optimización**

Para mejorar la legibilidad en las visualizaciones, se utilizan las siguientes abreviaturas para cada técnica de optimización:

* **STD** - Estándar (Standard)
* **MP** - Precisión Mixta (Mixed Precision)
* **GA256-128** - Acumulación de Gradientes (BS Efectivo: 256-128)
* **GA256-64** - Acumulación de Gradientes (BS Efectivo: 256-64)
* **Comb256-128** - Combinado (BS Efectivo: 256-128)
* **Comb256-64** - Combinado (BS Efectivo: 256-64)

* A continuación, un gráfico de barras comparará la **precisión final** (validación) de cada técnica.

In [ ]:
# Visualize the final accuracy to compare each optimization technique
helper_utils.plot_final_accuracy(results)

* El último compromiso (trade-off) a considerar es el uso de memoria de la GPU. Este gráfico visualiza el **pico de memoria** consumido durante cada ejecución, lo cual es fundamental para entender qué técnicas son las más eficientes en cuanto a recursos.

In [ ]:
# Visualize the peak memory usage for each optimization
helper_utils.plot_peak_memory(results)

## Análisis

El gráfico anterior muestra que la Precisión Mixta reduce el uso de memoria. De igual manera, el uso de tamaños de lote más pequeños (128 y 64) mientras se mantiene un tamaño de lote efectivo de 256 también disminuye el consumo de memoria. Finalmente, combinar la Acumulación de Gradientes con la Precisión Mixta reduce el uso de memoria aún más, todo esto sin comprometer la precisión de validación.

## Conclusión

¡Felicidades por completar el laboratorio! Has construido con éxito un marco de trabajo de benchmarking completo y lo has utilizado para medir el impacto de potentes técnicas de optimización de memoria. Los resultados de tus experimentos demuestran cómo reducir eficazmente el uso máximo de memoria (costos de infraestructura) manteniendo el rendimiento del modelo.


Además, has profundizado tus habilidades con Lightning. Has visto cómo utilizar un **`Callback`** personalizado para un monitoreo detallado y cómo aprovechar el **`Trainer`** para habilitar optimizaciones de memoria complejas mediante parámetros sencillos y declarativos.


Ahora cuentas con un enfoque práctico y basado en datos para tomar decisiones rentables sobre el entrenamiento de modelos. Al realizar sistemáticamente un benchmark de diferentes estrategias, puedes asegurarte de hacer el uso más eficiente de tu presupuesto de memoria de GPU, una habilidad esencial para gestionar los costos de infraestructura mientras escalas tu trabajo a modelos y conjuntos de datos más grandes.